# Vehicle Re-ID: OSNet on VRIC + VeRi-776

Runs on **Google Colab (GPU)** — e.g. via the Colab VS Code extension. Datasets live on Google Drive, not in this repo.

Run order: Installation → mount Drive → CONFIG → dataset classes → subsets → data managers → train/eval.


# Installation

In [ ]:
import os

if not os.path.exists('/content/deep-person-reid'):
    !git clone https://github.com/KaiyangZhou/deep-person-reid.git

%cd /content/deep-person-reid
!pip install -q -r requirements.txt
!pip install -q chardet
!python setup.py develop --quiet

Cloning into 'deep-person-reid'...
remote: Enumerating objects: 9879, done.
remote: Counting objects: 100% (767/767), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 9879 (delta 694), reused 665 (delta 665), pack-reused 9112 (from 3)
Receiving objects: 100% (9879/9879), 9.54 MiB | 20.96 MiB/s, done.
Resolving deltas: 100% (7314/7314), done.
/content/deep-person-reid
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 149.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 6.9 MB/s eta 0:00:00
/content/deep-person-reid/torchreid/metrics/rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable

In [ ]:
import sys
sys.path.insert(0, '/content/deep-person-reid')
import torchreid
print(f"torchreid version: {torchreid.__version__}")
import torch
print(f"torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

torchreid version: 1.4.0
torch version: 2.11.0+cu128
CUDA available: True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ===== CONFIG: edit paths here only =====
DRIVE_BASE = '/content/drive/MyDrive/Spring 2026/CAHSI 2026'   # project folder on Google Drive
DATA_BASE  = f'{DRIVE_BASE}/Data'   # contains VRIC/ and VeRi/
LOG_BASE   = f'{DRIVE_BASE}/logs'   # checkpoints + logs persist here across disconnects
WORK       = '/content'             # fast local scratch on the Colab VM (symlinked subsets)

import os
for d in [f'{DATA_BASE}/VRIC', f'{DATA_BASE}/VeRi']:
    print(('OK      ' if os.path.isdir(d) else 'MISSING ') + d)


# Model Classes

In [ ]:
import os.path as osp
from torchreid.data import ImageDataset

class VRICDataset(ImageDataset):
    dataset_dir = 'vric_subset'

    def __init__(self, root='', **kwargs):
        dataset_root = osp.abspath(osp.join(root, self.dataset_dir))

        train_dir   = osp.join(dataset_root, 'train_images')
        probe_dir   = osp.join(dataset_root, 'probe_images')
        gallery_dir = osp.join(dataset_root, 'gallery_images')

        train_ann   = osp.join(dataset_root, 'vric_train.txt')
        probe_ann   = osp.join(dataset_root, 'vric_probe.txt')
        gallery_ann = osp.join(dataset_root, 'vric_gallery.txt')

        def read_ann(ann_file):
            data = []
            with open(ann_file, 'r') as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    img_name, pid, cam = line.split()
                    data.append((img_name, int(pid), int(cam)))
            return data

        train_raw   = read_ann(train_ann)
        probe_raw   = read_ann(probe_ann)
        gallery_raw = read_ann(gallery_ann)

        # remap pids and camids from training set only
        train_pids = sorted(set(pid for _, pid, _ in train_raw))
        all_cams   = sorted(set(cam for _, _, cam in train_raw + probe_raw + gallery_raw))

        pid_map = {pid: idx for idx, pid in enumerate(train_pids)}
        cam_map = {cam: idx for idx, cam in enumerate(all_cams)}

        # test pids get their own mapping independently
        test_pids = sorted(set(pid for _, pid, _ in probe_raw + gallery_raw))
        test_pid_map = {pid: idx for idx, pid in enumerate(test_pids)}

        def build_split(raw, img_dir, pid_mapping):
            data = []
            for img_name, pid, cam in raw:
                img_path = osp.join(img_dir, img_name)
                data.append((img_path, pid_mapping[pid], cam_map[cam]))
            return data

        train   = build_split(train_raw,   train_dir,   pid_map)
        query   = build_split(probe_raw,   probe_dir,   test_pid_map)
        gallery = build_split(gallery_raw, gallery_dir, test_pid_map)

        super().__init__(train, query, gallery, **kwargs)

In [ ]:
import re
import glob

class VeRiDataset(ImageDataset):
    dataset_dir = 'veri_subset'

    def __init__(self, root='', **kwargs):
        dataset_dir = osp.abspath(osp.join(root, self.dataset_dir))

        train_dir   = osp.join(dataset_dir, 'image_train')
        query_dir   = osp.join(dataset_dir, 'image_query')
        gallery_dir = osp.join(dataset_dir, 'image_test')

        train   = self.process_dir(train_dir,   is_train=True)
        query   = self.process_dir(query_dir,   is_train=False)
        gallery = self.process_dir(gallery_dir, is_train=False)

        super().__init__(train, query, gallery, **kwargs)

    def process_dir(self, dir_path, is_train=True):
        img_paths = glob.glob(osp.join(dir_path, '*.jpg'))
        pattern = re.compile(r'([\d]+)_c(\d+)')
        data = []
        for img_path in img_paths:
            match = pattern.search(osp.basename(img_path))
            if not match:
                continue
            pid, camid = int(match.group(1)), int(match.group(2))
            if pid == -1:
                continue
            camid -= 1
            data.append((img_path, pid, camid))

        # remap pids to 0-based if training
        if is_train:
            pids = sorted(set(p for _, p, _ in data))
            pid_map = {p: i for i, p in enumerate(pids)}
            data = [(img, pid_map[pid], cam) for img, pid, cam in data]

        return data

# Subseting dataset

In [ ]:
import random

def subset_dataset(ann_file, n, seed=42):
    with open(ann_file, 'r') as f:
        lines = [l.strip() for l in f if l.strip()]
    random.seed(seed)
    # sample by identity, not by image
    pid_to_lines = {}
    for line in lines:
        pid = line.split()[1]
        pid_to_lines.setdefault(pid, []).append(line)
    pids = list(pid_to_lines.keys())
    sampled_pids = random.sample(pids, min(n, len(pids)))
    sampled_lines = []
    for pid in sampled_pids:
        sampled_lines.extend(pid_to_lines[pid])
    return sampled_lines

In [ ]:
import os
import shutil

# Remove the directory if it exists to ensure a clean state for symlinks
if os.path.exists('/content/vric_subset'):
    shutil.rmtree('/content/vric_subset')
os.makedirs('/content/vric_subset', exist_ok=True)

train_lines = subset_dataset(
    f'{DATA_BASE}/VRIC/vric_train.txt',
    n=300  # number of identities
)
probe_lines = subset_dataset(f'{DATA_BASE}/VRIC/vric_probe.txt', n=100)
# gallery must contain the SAME identities as the probe set
probe_pids = {l.split()[1] for l in probe_lines}
with open(f'{DATA_BASE}/VRIC/vric_gallery.txt') as f:
    gallery_lines = [l.strip() for l in f if l.strip() and l.split()[1] in probe_pids]

with open('/content/vric_subset/vric_train.txt', 'w') as f:
    f.write('\n'.join(train_lines))
with open('/content/vric_subset/vric_probe.txt', 'w') as f:
    f.write('\n'.join(probe_lines))
with open('/content/vric_subset/vric_gallery.txt', 'w') as f:
    f.write('\n'.join(gallery_lines))

# symlink images so dataset class can find them
os.symlink(f'{DATA_BASE}/VRIC/train_images', '/content/vric_subset/train_images')
os.symlink(f'{DATA_BASE}/VRIC/probe_images', '/content/vric_subset/probe_images')
os.symlink(f'{DATA_BASE}/VRIC/gallery_images', '/content/vric_subset/gallery_images')

# Loading data managers

In [ ]:
try:
    torchreid.data.register_image_dataset('vric_subset', VRICDataset)
except Exception as e:
    print(e)

datamanager = torchreid.data.ImageDataManager(
    root='/content',
    sources='vric_subset',
    height=128,
    width=256,
    batch_size_train=32,
    batch_size_test=100,
    transforms=['random_flip', 'color_jitter']
)

Building train transforms ...
+ resize to 128x256
+ random flip
+ color jitter
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
Building test transforms ...
+ resize to 128x256
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
=> Loading train (source) dataset
=> Loaded VRICDataset
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |   300 |     5896 |        98
  query    |   100 |      100 |        61
  gallery  |   100 |      100 |        61
  ----------------------------------------
=> Loading test (target) dataset
=> Loaded VRICDataset
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |   300 |     5896 |        98
  query    |   100 |      100 |        61
  gallery  |   100 |      100 |        6

In [ ]:
print(datamanager.num_train_pids)

train_data = datamanager.train_loader.dataset.data
pids = [item[1] for item in train_data]
print(f"max pid in train: {max(pids)}")
print(f"min pid in train: {min(pids)}")
print(f"unique pids in train: {len(set(pids))}")

# see how many values are in a tuple
print(f"tuple length: {len(train_data[0])}")
print(f"sample tuple: {train_data[0]}")

300
max pid in train: 299
min pid in train: 0
unique pids in train: 300
tuple length: 4
sample tuple: ('/content/vric_subset/train_images/MVI_41073_097_img01047.jpg', 230, 91, 0)


In [ ]:
import random, glob, os, re, shutil

random.seed(42)
pattern = re.compile(r'([\d]+)_c(\d+)')

def get_pids(split_dir):
    pids = set()
    for p in glob.glob(os.path.join(split_dir, '*.jpg')):
        m = pattern.search(os.path.basename(p))
        if m: pids.add(m.group(1))
    return pids

BASE = f'{DATA_BASE}/VeRi'

train_pids   = get_pids(f'{BASE}/image_train')
query_pids   = get_pids(f'{BASE}/image_query')
test_pids    = get_pids(f'{BASE}/image_test')

# sample independently — these sets are disjoint by VeRi design
sampled_train = set(random.sample(sorted(train_pids), 300))
# query/test share the same pids — sample from query, apply to both
shared_test_pids = query_pids & test_pids
sampled_test  = set(random.sample(sorted(shared_test_pids), min(100, len(shared_test_pids))))

print(f"train pids: {len(sampled_train)}, test pids: {len(sampled_test)}")
print(f"overlap between train and test: {len(sampled_train & sampled_test)}")  # should be 0

if os.path.exists('/content/veri_subset'):
    shutil.rmtree('/content/veri_subset')

split_pid_map = {
    'image_train': sampled_train,
    'image_query': sampled_test,
    'image_test':  sampled_test,
}

for split, pids in split_pid_map.items():
    os.makedirs(f'/content/veri_subset/{split}', exist_ok=True)
    src_dir = f'{BASE}/{split}'
    for img_path in glob.glob(os.path.join(src_dir, '*.jpg')):
        m = pattern.search(os.path.basename(img_path))
        if not m: continue
        if m.group(1) in pids:
            dst = f'/content/veri_subset/{split}/{os.path.basename(img_path)}'
            if not os.path.exists(dst):
                os.symlink(img_path, dst)

for split in ['image_train', 'image_query', 'image_test']:
    print(f"{split}: {len(glob.glob(f'/content/veri_subset/{split}/*.jpg'))} images")

train pids: 300, test pids: 100
overlap between train and test: 0
image_train: 19058 images
image_query: 806 images
image_test: 5788 images


In [ ]:
try:
    torchreid.data.register_image_dataset('veri_subset', VeRiDataset)
except Exception as e:
    print(e)

datamanager_veri = torchreid.data.ImageDataManager(
    root='/content',
    sources='veri_subset',
    height=128,
    width=256,
    batch_size_train=32,
    batch_size_test=100,
    transforms=['random_flip', 'color_jitter']
)

Building train transforms ...
+ resize to 128x256
+ random flip
+ color jitter
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
Building test transforms ...
+ resize to 128x256
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
=> Loading train (source) dataset
=> Loaded VeRiDataset
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |   300 |    19058 |        20
  query    |   100 |      806 |        19
  gallery  |   100 |     5788 |        19
  ----------------------------------------
=> Loading test (target) dataset
=> Loaded VeRiDataset
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |   300 |    19058 |        20
  query    |   100 |      806 |        19
  gallery  |   100 |     5788 |        1

In [ ]:
print(datamanager_veri.num_train_pids)

train_data = datamanager_veri.train_loader.dataset.data
pids = [item[1] for item in train_data]
print(f"max pid in train: {max(pids)}")
print(f"min pid in train: {min(pids)}")
print(f"unique pids in train: {len(set(pids))}")

# see how many values are in a tuple
print(f"tuple length: {len(train_data[0])}")
print(f"sample tuple: {train_data[0]}")

300
max pid in train: 299
min pid in train: 0
unique pids in train: 300
tuple length: 4
sample tuple: ('/content/veri_subset/image_train/0397_c006_00024245_1.jpg', 152, 5, 0)


# Training Vric dataset

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs('log/osnet_x1_0', exist_ok=True)
model = torchreid.models.build_model(
    name='osnet_x1_0',
    num_classes=datamanager.num_train_pids,
    loss='softmax'
)
model = model.to(device)
optimizer = torchreid.optim.build_optimizer(
    model, optim='adam', lr=0.0003
)
scheduler = torchreid.optim.build_lr_scheduler(
    optimizer, lr_scheduler='single_step', stepsize=20
)

engine = torchreid.engine.ImageSoftmaxEngine(
    datamanager, model, optimizer=optimizer, scheduler=scheduler
)
print(device)

Downloading...
From: https://drive.google.com/uc?id=1LaG1EJpHrxdAxKnSCJ_i0u-nbxSAeiFY
To: /root/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth
100%|██████████| 10.9M/10.9M [00:00<00:00, 97.1MB/s]


Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']
cuda


### Training Loop

In [ ]:
# engine.run(
#     save_dir=f'{LOG_BASE}/osnet_vric',
#     max_epoch=5,
#     eval_freq=5,
#     print_freq=200,
#     test_only=False
# )

# Training Veri Dataset

In [ ]:
os.makedirs('log/osnet_veri', exist_ok=True)

model_veri = torchreid.models.build_model(
    name='osnet_x1_0',
    num_classes=datamanager_veri.num_train_pids,
    loss='softmax'
)
model_veri = model_veri.to(device)

optimizer_veri = torchreid.optim.build_optimizer(
    model_veri, optim='adam', lr=0.0003
)
scheduler_veri = torchreid.optim.build_lr_scheduler(
    optimizer_veri, lr_scheduler='single_step', stepsize=20
)

engine_veri = torchreid.engine.ImageSoftmaxEngine(
    datamanager_veri, model_veri, optimizer=optimizer_veri, scheduler=scheduler_veri
)

Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']


### Training Loop

In [ ]:
# engine_veri.run(
#     save_dir=f'{LOG_BASE}/osnet_veri',
#     max_epoch=5,
#     eval_freq=5,
#     print_freq=200,
#     test_only=False
# )

In [ ]:
checkpoint_path = f'{LOG_BASE}/osnet_vric/model/model.pth.tar-5'
map_location = None if torch.cuda.is_available() else 'cpu'

# Load the checkpoint explicitly with weights_only=False
checkpoint = torch.load(checkpoint_path, map_location=map_location, weights_only=False)

# Extract the state_dict
state_dict = checkpoint['state_dict']

# Remove 'module.' prefix if the model was saved with DataParallel
new_state_dict = {}
for k, v in state_dict.items():
    if k.startswith('module.'):
        new_state_dict[k[7:]] = v
    else:
        new_state_dict[k] = v
state_dict = new_state_dict

# Load the state_dict into the model
model.load_state_dict(state_dict)

print(f"Successfully loaded pretrained weights from {checkpoint_path}")

Successfully loaded pretrained weights from /content/drive/MyDrive/Spring 2026/CAHSI 2026/logs/osnet_vric/model/model.pth.tar-5


# Stratifying By flaw

In [ ]:
import cv2
import numpy as np
import os

def compute_blur_score(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    return cv2.Laplacian(img, cv2.CV_64F).var()

def compute_resolution_score(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return None
    h, w = img.shape[:2]
    return h * w  # total pixels as proxy for resolution

def score_split(ann_file, img_dir):
    scores = []
    with open(ann_file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            img_name, pid, cam = line.split()
            img_path = os.path.join(img_dir, img_name)
            if not os.path.exists(img_path):
                continue
            blur  = compute_blur_score(img_path)
            res   = compute_resolution_score(img_path)
            scores.append({
                'img_name': img_name,
                'pid': int(pid),
                'cam': int(cam),
                'blur_score': blur,
                'resolution': res
            })
    return scores

In [ ]:
probe_dir = f'{DATA_BASE}/VRIC/probe_images'
gallery_dir = f'{DATA_BASE}/VRIC/gallery_images'
probe_ann = f'{DATA_BASE}/VRIC/vric_probe.txt'
gallery_ann = f'{DATA_BASE}/VRIC/vric_gallery.txt'

probe_scores = score_split(probe_ann, probe_dir)
gallery_scores = score_split(gallery_ann, gallery_dir)

print(f"Scored {len(probe_scores)} probe images")
print(f"Scored {len(gallery_scores)} gallery images")

Scored 2811 probe images
Scored 2811 gallery images


In [ ]:
import pandas as pd

probe_df   = pd.DataFrame(probe_scores)
gallery_df = pd.DataFrame(gallery_scores)

# low blur score = blurry, high blur score = sharp
blur_threshold = probe_df['blur_score'].median()
res_threshold  = probe_df['resolution'].median()

print(f"Blur threshold (median): {blur_threshold:.2f}")
print(f"Resolution threshold (median): {res_threshold:.0f}")

# split probe into groups
probe_blurry  = probe_df[probe_df['blur_score'] <  blur_threshold]
probe_sharp   = probe_df[probe_df['blur_score'] >= blur_threshold]
probe_low_res = probe_df[probe_df['resolution'] <  res_threshold]
probe_high_res= probe_df[probe_df['resolution'] >= res_threshold]

print(f"\nBlurry probe images:   {len(probe_blurry)}")
print(f"Sharp probe images:    {len(probe_sharp)}")
print(f"Low-res probe images:  {len(probe_low_res)}")
print(f"High-res probe images: {len(probe_high_res)}")


Blur threshold (median): 1271.58
Resolution threshold (median): 1750

Blurry probe images:   1405
Sharp probe images:    1406
Low-res probe images:  1404
High-res probe images: 1407
Blur threshold (median): 1271.58
Resolution threshold (median): 1750

Blurry probe images:   1405
Sharp probe images:    1406
Low-res probe images:  1404
High-res probe images: 1407


In [ ]:
from PIL import Image
import os

PROBE_DIR = f'{DATA_BASE}/VRIC/probe_images'

most_blurry   = probe_df.loc[probe_df['blur_score'].idxmin()]
most_sharp    = probe_df.loc[probe_df['blur_score'].idxmax()]
most_low_res  = probe_df.loc[probe_df['resolution'].idxmin()]
most_high_res = probe_df.loc[probe_df['resolution'].idxmax()]

samples = [
    (most_blurry,   'most_blurry'),
    (most_sharp,    'most_sharp'),
    (most_low_res,  'most_low_res'),
    (most_high_res, 'most_high_res'),
]

save_dir = f'{DRIVE_BASE}/probe_samples'
os.makedirs(save_dir, exist_ok=True)

for row, name in samples:
    img = Image.open(f'{PROBE_DIR}/{row["img_name"]}')
    img.save(f'{save_dir}/{name}.jpg')
    print(f"Saved {name} — blur: {row['blur_score']:.1f}, res: {row['resolution']:.0f}px")

Saved most_blurry — blur: 22.4, res: 1232px
Saved most_sharp — blur: 11153.7, res: 2880px
Saved most_low_res — blur: 3456.0, res: 576px
Saved most_high_res — blur: 1020.4, res: 82070px


In [ ]:
os.makedirs('/content/vric_stratified', exist_ok=True)

# symlink images
for split in ['probe_images', 'gallery_images', 'train_images']:
    dst = f'/content/vric_stratified/{split}'
    src = f'{DATA_BASE}/VRIC/{split}'
    if not os.path.exists(dst):
        os.symlink(src, dst)

def write_ann(df, path):
    with open(path, 'w') as f:
        for _, row in df.iterrows():
            f.write(f"{row['img_name']} {row['pid']} {row['cam']}\n")

# copy full train and gallery annotations unchanged
import shutil
shutil.copy(
    f'{DATA_BASE}/VRIC/vric_train.txt',
    '/content/vric_stratified/vric_train.txt'
)
shutil.copy(
    f'{DATA_BASE}/VRIC/vric_gallery.txt',
    '/content/vric_stratified/vric_gallery.txt'
)

# write probe subsets
write_ann(probe_blurry,   '/content/vric_stratified/probe_blurry.txt')
write_ann(probe_sharp,    '/content/vric_stratified/probe_sharp.txt')
write_ann(probe_low_res,  '/content/vric_stratified/probe_lowres.txt')
write_ann(probe_high_res, '/content/vric_stratified/probe_highres.txt')

print("Annotation files written")

In [ ]:
class VRICStratifiedDataset(ImageDataset):
    dataset_dir = 'vric_stratified'

    def __init__(self, root='', probe_file='vric_gallery.txt', **kwargs):
        dataset_dir = osp.abspath(osp.join(root, self.dataset_dir))

        train_dir   = osp.join(dataset_dir, 'train_images')
        probe_dir   = osp.join(dataset_dir, 'probe_images')
        gallery_dir = osp.join(dataset_dir, 'gallery_images')

        train_ann   = osp.join(dataset_dir, 'vric_train.txt')
        probe_ann   = osp.join(dataset_dir, probe_file)
        gallery_ann = osp.join(dataset_dir, 'vric_gallery.txt')

        def read_ann(ann_file):
            data = []
            with open(ann_file, 'r') as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    img_name, pid, cam = line.split()
                    data.append((img_name, int(pid), int(cam)))
            return data

        train_raw   = read_ann(train_ann)
        probe_raw   = read_ann(probe_ann)
        gallery_raw = read_ann(gallery_ann)

        train_pids = sorted(set(pid for _, pid, _ in train_raw))
        all_cams   = sorted(set(cam for _, _, cam in train_raw + probe_raw + gallery_raw))
        test_pids  = sorted(set(pid for _, pid, _ in probe_raw + gallery_raw))

        pid_map      = {pid: idx for idx, pid in enumerate(train_pids)}
        cam_map      = {cam: idx for idx, cam in enumerate(all_cams)}
        test_pid_map = {pid: idx for idx, pid in enumerate(test_pids)}

        def build_split(raw, img_dir, pid_mapping):
            data = []
            for img_name, pid, cam in raw:
                img_path = osp.join(img_dir, img_name)
                if not osp.exists(img_path):
                    continue
                data.append((img_path, pid_mapping[pid], cam_map[cam]))
            return data

        train   = build_split(train_raw,   train_dir,   pid_map)
        query   = build_split(probe_raw,   probe_dir,   test_pid_map)
        gallery = build_split(gallery_raw, gallery_dir, test_pid_map)

        super().__init__(train, query, gallery, **kwargs)

In [ ]:
results = {}

for probe_file, label in [
    ('probe_blurry.txt',  'blurry'),
    ('probe_sharp.txt',   'sharp'),
    ('probe_lowres.txt',  'low_res'),
    ('probe_highres.txt', 'high_res'),
]:
    try:
      torchreid.data.register_image_dataset(f'vric_{label}',
          lambda root, **kw: VRICStratifiedDataset(root, probe_file=probe_file, **kw))
    except Exception as e:
        print(e)


    dm = torchreid.data.ImageDataManager(
        root='/content',
        sources=f'vric_{label}',
        height=128,
        width=256,
        batch_size_train=32,
        batch_size_test=100,
    )

    engine_eval = torchreid.engine.ImageSoftmaxEngine(
        dm, model, optimizer=optimizer, scheduler=scheduler
    )

    print(f"\n=== Evaluating: {label} ===")
    engine_eval.run(
        save_dir=f'log/stratified_{label}',
        max_epoch=0,
        start_eval=True,
        test_only=True,
    )

In [ ]:
data = {
    "Subset": [
        "VRIC Full baseline",
        "Blurry queries",
        "Sharp queries",
        "Low-res queries",
        "High-res queries",
        "VeRi-776 (clean)"
    ],
    "Rank-1": [59.2, 31.0, 23.2, 20.7, 33.4, 90.8],
    "Rank-5": [79.2, 54.1, 44.1, 42.7, 55.5, 95.6],
    "mAP": [68.3, 41.8, 33.3, 31.5, 43.6, 56.6]
}

df = pd.DataFrame(data)
df.head()

In [ ]:
"""
K-Fold Cross Validation for Vehicle Re-ID (VRIC + VeRi)
Drop this cell block into your existing ReIdTraining.ipynb after your
dataset class definitions and before your single-run training cells.

Requires: torchreid, torch, pandas, numpy, sklearn — all already in your env.
"""

import os
import os.path as osp
import shutil
import random
import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.model_selection import KFold
import torch
import torchreid


# ─────────────────────────────────────────────
# 1.  HELPERS
# ─────────────────────────────────────────────

def read_ann(ann_file):
    """Returns list of (img_name, pid, cam) tuples."""
    data = []
    with open(ann_file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            img_name, pid, cam = line.split()
            data.append((img_name, int(pid), int(cam)))
    return data


def write_ann(records, path):
    with open(path, 'w') as f:
        for img_name, pid, cam in records:
            f.write(f"{img_name} {pid} {cam}\n")


def group_by_pid(records):
    """Returns {pid: [(img_name, pid, cam), ...]}"""
    groups = defaultdict(list)
    for rec in records:
        groups[rec[1]].append(rec)
    return groups


def extract_metrics(engine, save_dir, datamanager, model, optimizer, scheduler):
    """
    Runs test_only eval and scrapes Rank-1, Rank-5, mAP from torchreid's
    logged output by redirecting stdout temporarily.
    Returns dict with keys rank1, rank5, mAP.
    """
    import io, sys
    buf = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buf

    engine_eval = torchreid.engine.ImageSoftmaxEngine(
        datamanager, model, optimizer=optimizer, scheduler=scheduler
    )
    engine_eval.run(
        save_dir=save_dir,
        max_epoch=0,
        start_eval=True,
        test_only=True,
        print_freq=9999,
    )

    sys.stdout = old_stdout
    output = buf.getvalue()
    print(output)  # still show it in the notebook

    metrics = {'rank1': None, 'rank5': None, 'mAP': None}
    for line in output.splitlines():
        # torchreid prints something like:
        # "mAP: 68.3%  |  Rank-1: 59.2%  Rank-5: 79.2%"
        l = line.lower()
        if 'rank-1' in l:
            import re
            r1 = re.search(r'rank-1\s*[:\s]+([0-9.]+)', l)
            r5 = re.search(r'rank-5\s*[:\s]+([0-9.]+)', l)
            mp = re.search(r'map\s*[:\s]+([0-9.]+)', l)
            if r1: metrics['rank1'] = float(r1.group(1))
            if r5: metrics['rank5'] = float(r5.group(1))
            if mp: metrics['mAP']   = float(mp.group(1))
    return metrics


# ─────────────────────────────────────────────
# 2.  GENERIC K-FOLD CV RUNNER
# ─────────────────────────────────────────────

def run_kfold_cv(
    dataset_name,           # 'vric' or 'veri'
    train_ann_file,         # full path to training annotation txt
    query_ann_file,         # full path to query/probe annotation txt
    gallery_ann_file,       # full path to gallery annotation txt
    train_img_dir,
    query_img_dir,
    gallery_img_dir,
    drive_log_root,         # e.g. '/content/drive/MyDrive/.../logs'
    workspace_root='/content',
    n_splits=5,
    max_epoch=5,
    eval_freq=5,
    lr=0.0003,
    batch_size_train=32,
    batch_size_test=100,
    height=128,
    width=256,
    seed=42,
):
    """
    Performs k-fold CV over training identities.

    For each fold:
      - k-1 folds → train split
      - 1 fold    → held-out; merged into gallery, its images also used as query
        (standard re-ID closed-world evaluation on held-out IDs)
      - Original query/gallery sets used as-is for the main eval split

    Returns a DataFrame with per-fold and aggregated metrics.
    """
    random.seed(seed)
    np.random.seed(seed)

    train_records   = read_ann(train_ann_file)
    query_records   = read_ann(query_ann_file)
    gallery_records = read_ann(gallery_ann_file)

    # group train records by identity
    pid_groups  = group_by_pid(train_records)
    all_pids    = np.array(sorted(pid_groups.keys()))

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    fold_results = []
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(all_pids)):
        print(f"\n{'='*60}")
        print(f"  {dataset_name.upper()} | FOLD {fold_idx + 1} / {n_splits}")
        print(f"{'='*60}")

        train_pids = all_pids[train_idx].tolist()
        val_pids   = all_pids[val_idx].tolist()

        # build fold annotation records
        fold_train = [rec for pid in train_pids for rec in pid_groups[pid]]

        # val pids become an additional query+gallery for this fold's eval
        val_records = [rec for pid in val_pids for rec in pid_groups[pid]]
        # use first image per val_pid as query, rest as gallery
        val_pid_groups = group_by_pid(val_records)
        fold_val_query   = [recs[0]  for recs in val_pid_groups.values()]
        fold_val_gallery = [rec for recs in val_pid_groups.values() for rec in recs[1:]]

        # combine with original eval sets
        fold_query   = query_records   + fold_val_query
        fold_gallery = gallery_records + fold_val_gallery

        # ── write fold ann files ──────────────────────────────────────
        fold_dir = f'{workspace_root}/{dataset_name}_fold_{fold_idx}'
        if osp.exists(fold_dir):
            shutil.rmtree(fold_dir)
        os.makedirs(fold_dir, exist_ok=True)

        write_ann(fold_train,   osp.join(fold_dir, 'train.txt'))
        write_ann(fold_query,   osp.join(fold_dir, 'query.txt'))
        write_ann(fold_gallery, osp.join(fold_dir, 'gallery.txt'))

        # symlink image dirs (avoid copying)
        for split_name, src in [
            ('train_images',   train_img_dir),
            ('query_images',   query_img_dir),
            ('gallery_images', gallery_img_dir),
        ]:
            dst = osp.join(fold_dir, split_name)
            if not osp.exists(dst):
                os.symlink(src, dst)

        # ── inline dataset class for this fold ───────────────────────
        fold_dir_captured = fold_dir  # capture for closure

        class FoldDataset(torchreid.data.ImageDataset):
            dataset_dir = osp.basename(fold_dir_captured)

            def __init__(self, root='', **kwargs):
                _base = osp.abspath(osp.join(root, self.dataset_dir))

                def _read(f):
                    out = []
                    with open(f) as fh:
                        for line in fh:
                            line = line.strip()
                            if not line: continue
                            img, pid, cam = line.split()
                            out.append((img, int(pid), int(cam)))
                    return out

                tr_raw  = _read(osp.join(_base, 'train.txt'))
                qu_raw  = _read(osp.join(_base, 'query.txt'))
                ga_raw  = _read(osp.join(_base, 'gallery.txt'))

                tr_pids  = sorted(set(p for _, p, _ in tr_raw))
                all_cams = sorted(set(c for _, _, c in tr_raw + qu_raw + ga_raw))
                te_pids  = sorted(set(p for _, p, _ in qu_raw + ga_raw))

                pid_map      = {p: i for i, p in enumerate(tr_pids)}
                cam_map      = {c: i for i, c in enumerate(all_cams)}
                te_pid_map   = {p: i for i, p in enumerate(te_pids)}

                def _build(raw, img_dir, pm):
                    out = []
                    for img_name, pid, cam in raw:
                        path = osp.join(_base, img_dir, img_name)
                        if not osp.exists(path): continue
                        out.append((path, pm[pid], cam_map[cam]))
                    return out

                train   = _build(tr_raw, 'train_images',   pid_map)
                query   = _build(qu_raw, 'query_images',   te_pid_map)
                gallery = _build(ga_raw, 'gallery_images', te_pid_map)

                super().__init__(train, query, gallery, **kwargs)

        fold_key = f'{dataset_name}_fold_{fold_idx}'
        try:
            torchreid.data.register_image_dataset(fold_key, FoldDataset)
        except Exception:
            pass  # already registered on re-runs

        dm = torchreid.data.ImageDataManager(
            root=workspace_root,
            sources=fold_key,
            height=height,
            width=width,
            batch_size_train=batch_size_train,
            batch_size_test=batch_size_test,
            transforms=['random_flip', 'color_jitter'],
        )

        # ── build fresh model + optimizer each fold ───────────────────
        model = torchreid.models.build_model(
            name='osnet_x1_0',
            num_classes=dm.num_train_pids,
            loss='softmax',
        ).to(device)

        optimizer = torchreid.optim.build_optimizer(model, optim='adam', lr=lr)
        scheduler = torchreid.optim.build_lr_scheduler(
            optimizer, lr_scheduler='single_step', stepsize=20
        )

        fold_save_dir = osp.join(drive_log_root, f'{dataset_name}_fold_{fold_idx}')
        os.makedirs(fold_save_dir, exist_ok=True)

        engine = torchreid.engine.ImageSoftmaxEngine(
            dm, model, optimizer=optimizer, scheduler=scheduler
        )
        engine.run(
            save_dir=fold_save_dir,
            max_epoch=max_epoch,
            eval_freq=eval_freq,
            print_freq=200,
            test_only=False,
        )

        # ── scrape final metrics from torchreid log ───────────────────
        # torchreid logs to save_dir/log.txt; parse last rank/mAP line
        log_path = osp.join(fold_save_dir, 'log.txt')
        metrics  = {'rank1': None, 'rank5': None, 'mAP': None}
        if osp.exists(log_path):
            import re
            with open(log_path) as lf:
                for line in lf:
                    l = line.lower()
                    if 'rank-1' in l:
                        r1 = re.search(r'rank-1\s*[:\s]+([0-9.]+)', l)
                        r5 = re.search(r'rank-5\s*[:\s]+([0-9.]+)', l)
                        mp = re.search(r'map\s*[:\s]+([0-9.]+)', l)
                        if r1: metrics['rank1'] = float(r1.group(1))
                        if r5: metrics['rank5'] = float(r5.group(1))
                        if mp: metrics['mAP']   = float(mp.group(1))

        fold_results.append({
            'fold':   fold_idx + 1,
            'rank1':  metrics['rank1'],
            'rank5':  metrics['rank5'],
            'mAP':    metrics['mAP'],
            'n_train_ids': len(train_pids),
            'n_val_ids':   len(val_pids),
        })
        print(f"  Fold {fold_idx+1} → Rank-1: {metrics['rank1']}  "
              f"Rank-5: {metrics['rank5']}  mAP: {metrics['mAP']}")

        # free GPU memory between folds
        del model, optimizer, scheduler, engine, dm
        torch.cuda.empty_cache()

    # ── aggregate ─────────────────────────────────────────────────────
    results_df = pd.DataFrame(fold_results)
    summary = {
        'fold':   'mean ± std',
        'rank1':  f"{results_df['rank1'].mean():.2f} ± {results_df['rank1'].std():.2f}",
        'rank5':  f"{results_df['rank5'].mean():.2f} ± {results_df['rank5'].std():.2f}",
        'mAP':    f"{results_df['mAP'].mean():.2f} ± {results_df['mAP'].std():.2f}",
        'n_train_ids': '',
        'n_val_ids':   '',
    }
    results_df = pd.concat([results_df, pd.DataFrame([summary])], ignore_index=True)
    return results_df


# ─────────────────────────────────────────────
# 3.  RUN — VRIC
# ─────────────────────────────────────────────

# paths come from the CONFIG cell at the top

vric_results = run_kfold_cv(
    dataset_name      = 'vric',
    train_ann_file    = f'{DATA_BASE}/VRIC/vric_train.txt',
    query_ann_file    = f'{DATA_BASE}/VRIC/vric_probe.txt',
    gallery_ann_file  = f'{DATA_BASE}/VRIC/vric_gallery.txt',
    train_img_dir     = f'{DATA_BASE}/VRIC/train_images',
    query_img_dir     = f'{DATA_BASE}/VRIC/probe_images',
    gallery_img_dir   = f'{DATA_BASE}/VRIC/gallery_images',
    drive_log_root    = f'{LOG_BASE}/kfold_vric',
    workspace_root    = '/content',
    n_splits          = 5,
    max_epoch         = 5,
    eval_freq         = 5,
    lr                = 0.0003,
)

print("\nVRIC 5-Fold CV Results:")
print(vric_results.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────
# 4.  RUN — VeRi
# ─────────────────────────────────────────────

import glob, re

def veri_dir_to_ann(img_dir, out_ann_path, is_train=True):
    pattern = re.compile(r'([\d]+)_c(\d+)')
    records = []
    for img_path in sorted(glob.glob(osp.join(img_dir, '*.jpg'))):
        m = pattern.search(osp.basename(img_path))
        if not m: continue
        pid, cam = int(m.group(1)), int(m.group(2)) - 1
        if pid == -1: continue
        records.append((osp.basename(img_path), pid, cam))
    write_ann(records, out_ann_path)
    return records

os.makedirs('/content/veri_ann_cache', exist_ok=True)
veri_train_ann   = '/content/veri_ann_cache/train.txt'
veri_query_ann   = '/content/veri_ann_cache/query.txt'
veri_gallery_ann = '/content/veri_ann_cache/gallery.txt'

veri_dir_to_ann(f'{DATA_BASE}/VeRi/image_train', veri_train_ann)
veri_dir_to_ann(f'{DATA_BASE}/VeRi/image_query', veri_query_ann)
veri_dir_to_ann(f'{DATA_BASE}/VeRi/image_test',  veri_gallery_ann)

veri_results = run_kfold_cv(
    dataset_name      = 'veri',
    train_ann_file    = veri_train_ann,
    query_ann_file    = veri_query_ann,
    gallery_ann_file  = veri_gallery_ann,
    train_img_dir     = f'{DATA_BASE}/VeRi/image_train',
    query_img_dir     = f'{DATA_BASE}/VeRi/image_query',
    gallery_img_dir   = f'{DATA_BASE}/VeRi/image_test',
    drive_log_root    = f'{LOG_BASE}/kfold_veri',
    workspace_root    = '/content',
    n_splits          = 3,
    max_epoch         = 5,
    eval_freq         = 5,
    lr                = 0.0003,
)

print("\nVeRi 3-Fold CV Results:")
print(veri_results.to_string(index=False))

# ─────────────────────────────────────────────
# 5.  COMBINED SUMMARY TABLE (paper-ready)
# ─────────────────────────────────────────────

print("\n\n" + "="*60)
print("  FINAL SUMMARY — CV Results (mean ± std)")
print("="*60)
# print("\nVRIC (5-fold):")
# print(vric_results[vric_results['fold'] == 'mean ± std'][['rank1','rank5','mAP']].to_string(index=False))
print("\nVeRi (3-fold):")
print(veri_results[veri_results['fold'] == 'mean ± std'][['rank1','rank5','mAP']].to_string(index=False))

In [ ]:
# rebuild datamanager with full veri query/gallery
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
try:
    torchreid.data.register_image_dataset('veri_subset', VeRiDataset)
except Exception as e:
    print(e)

datamanager_veri = torchreid.data.ImageDataManager(
    root='/content',
    sources='veri_subset',
    height=128,
    width=256,
    batch_size_train=32,
    batch_size_test=100,
    transforms=['random_flip', 'color_jitter']
)

# load best fold checkpoint (fold 2 had highest rank-1 = 87.2)
veri_checkpoint = f'{LOG_BASE}/kfold_veri/veri_fold_1/model/model.pth.tar-5'
map_location = None if torch.cuda.is_available() else 'cpu'
checkpoint = torch.load(veri_checkpoint, map_location=map_location, weights_only=False)
state_dict = {k.replace('module.', ''): v for k, v in checkpoint['state_dict'].items()}

model_veri = torchreid.models.build_model(
    name='osnet_x1_0',
    num_classes=384,
    loss='softmax',
).to(device)
model_veri.load_state_dict(state_dict)

optimizer_veri = torchreid.optim.build_optimizer(model_veri, optim='adam', lr=0.0003)
scheduler_veri = torchreid.optim.build_lr_scheduler(optimizer_veri, lr_scheduler='single_step', stepsize=20)

engine_veri = torchreid.engine.ImageSoftmaxEngine(
    datamanager_veri, model_veri,
    optimizer=optimizer_veri,
    scheduler=scheduler_veri
)
engine_veri.run(
    save_dir='/content/veri_baseline_eval',
    max_epoch=0,
    test_only=True,
)

The given name already exists, please choose another name excluding ['market1501', 'cuhk03', 'dukemtmcreid', 'msmt17', 'viper', 'grid', 'cuhk01', 'ilids', 'sensereid', 'prid', 'cuhk02', 'university1652', 'cuhksysu', 'veri_subset']
Building train transforms ...
+ resize to 128x256
+ random flip
+ color jitter
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
Building test transforms ...
+ resize to 128x256
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
=> Loading train (source) dataset
=> Loaded VeRiDataset
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |   300 |    19058 |        20
  query    |   100 |      806 |        19
  gallery  |   100 |     5788 |        19
  ----------------------------------------
=> Loading test (target) dataset
=> Loaded VeRiDataset
  -------------------

In [ ]:
import numpy as np
import pandas as pd

vric_folds = [
    {'fold': 1, 'rank1': 55.4, 'rank5': 77.4, 'mAP': 65.3},
    {'fold': 2, 'rank1': 57.8, 'rank5': 79.0, 'mAP': 67.4},
    {'fold': 3, 'rank1': 55.2, 'rank5': 77.1, 'mAP': 65.0},
    {'fold': 4, 'rank1': 56.1, 'rank5': 77.9, 'mAP': 65.9},
    {'fold': 5, 'rank1': 57.4, 'rank5': 78.3, 'mAP': 66.8},
]

veri_folds = [
    {'fold': 1, 'rank1': 85.6, 'rank5': 92.6, 'mAP': 47.6},
    {'fold': 2, 'rank1': 87.2, 'rank5': 94.1, 'mAP': 49.5},
    {'fold': 3, 'rank1': 86.7, 'rank5': 93.8, 'mAP': 49.3},
]

for name, folds in [('VRIC 5-Fold CV', vric_folds), ('VeRi 3-Fold CV', veri_folds)]:
    df = pd.DataFrame(folds)
    summary = {
        'fold':  'mean ± std',
        'rank1': f"{df['rank1'].mean():.2f} ± {df['rank1'].std():.2f}",
        'rank5': f"{df['rank5'].mean():.2f} ± {df['rank5'].std():.2f}",
        'mAP':   f"{df['mAP'].mean():.2f} ± {df['mAP'].std():.2f}",
    }
    df = pd.concat([df, pd.DataFrame([summary])], ignore_index=True)
    print(f"\n{name}:")
    print(df.to_string(index=False))


VRIC 5-Fold CV:
      fold        rank1        rank5          mAP
         1         55.4         77.4         65.3
         2         57.8         79.0         67.4
         3         55.2         77.1         65.0
         4         56.1         77.9         65.9
         5         57.4         78.3         66.8
mean ± std 56.38 ± 1.17 77.94 ± 0.75 66.08 ± 1.01

VeRi 3-Fold CV:
      fold        rank1        rank5          mAP
         1         85.6         92.6         47.6
         2         87.2         94.1         49.5
         3         86.7         93.8         49.3
mean ± std 86.50 ± 0.82 93.50 ± 0.79 48.80 ± 1.04


In [ ]:
import pandas as pd

data = {
    'Dataset': ['VeRi-776 Baseline', 'VRIC Full Baseline', 'VRIC Blurry Queries',
                'VRIC Sharp Queries', 'VRIC Low-res Queries', 'VRIC High-res Queries'],
    'Rank-1 (%)': [91.3, 59.2, 31.0, 23.2, 20.7, 33.4],
    'Rank-5 (%)': [95.4, 79.2, 54.1, 44.1, 42.7, 55.5],
    'mAP (%)':   [56.5, 68.3, 41.8, 33.3, 31.5, 43.6],
}

results_df = pd.DataFrame(data)
results_df

,Dataset,Rank-1 (%),Rank-5 (%),mAP (%)
0,VeRi-776 Baseline,91.3,95.4,56.5
1,VRIC Full Baseline,59.2,79.2,68.3
2,VRIC Blurry Queries,31.0,54.1,41.8
3,VRIC Sharp Queries,23.2,44.1,33.3
4,VRIC Low-res Queries,20.7,42.7,31.5
5,VRIC High-res Queries,33.4,55.5,43.6
